# 🏥 MediGemma — Notebook 01: Knowledge Base Setup & RAG Pipeline Validation

This notebook covers:
- **Environment & dependency check**
- **Knowledge base ingestion** via `MedicalKnowledgeBase`
- **Vector store population** via `MedicalVectorStore`
- **RAG pipeline validation** — similarity search, hybrid search, retriever


## 📦 Cell 1 — Install Dependencies

In [ ]:
%pip install --upgrade pip setuptools wheel
%pip install -e
%pip install python-frontmatter 
%pip install pypdf

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import pkgutil
print([p.name for p in pkgutil.iter_modules()])

['_asyncio', '_bz2', '_ctypes', '_decimal', '_elementtree', '_hashlib', '_lzma', '_multiprocessing', '_overlapped', '_queue', '_remote_debugging', '_socket', '_sqlite3', '_ssl', '_tkinter', '_uuid', '_wmi', '_zoneinfo', '_zstd', 'pyexpat', 'select', 'unicodedata', 'winsound', '__future__', '__hello__', '__phello__', '_aix_support', '_android_support', '_apple_support', '_ast_unparse', '_collections_abc', '_colorize', '_compat_pickle', '_ios_support', '_markupbase', '_opcode_metadata', '_osx_support', '_py_abc', '_py_warnings', '_pydatetime', '_pydecimal', '_pyio', '_pylong', '_pyrepl', '_sitebuiltins', '_strptime', '_threading_local', '_weakrefset', 'abc', 'annotationlib', 'antigravity', 'argparse', 'ast', 'asyncio', 'base64', 'bdb', 'bisect', 'bz2', 'cProfile', 'calendar', 'cmd', 'code', 'codecs', 'codeop', 'collections', 'colorsys', 'compileall', 'compression', 'concurrent', 'configparser', 'contextlib', 'contextvars', 'copy', 'copyreg', 'csv', 'ctypes', 'curses', 'dataclasses', 'dat

In [2]:
%pip install langchain_chroma

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Install all required packages
# Run once; comment out after first successful install

%pip install -q \
    chromadb \
    langchain \
    langchain-huggingface \
    langchain-community \
    langchain-huggingface \
    langchain-text-splitters \
    sentence-transformers \
    torch --index-url https://download.pytorch.org/whl/cpu

print("✅ Dependencies installed")

Note: you may need to restart the kernel to use updated packages.
✅ Dependencies installed


ERROR: Could not find a version that satisfies the requirement langchain-chroma (from versions: none)
ERROR: No matching distribution found for langchain-chroma


## 🔧 Cell 2 — Path Setup & Project Root

In [2]:
import sys
import os
from pathlib import Path

# ── Adjust this to your actual project root ──────────────────────────────────
PROJECT_ROOT = Path("../").resolve()   # notebooks/ is one level below root
#sys.path.insert(0, str(PROJECT_ROOT))

print(f"📁 Project root  : {PROJECT_ROOT}")
print(f"📁 Working dir   : {Path.cwd()}")
print(f"🐍 Python version: {sys.version.split()[0]}")

# Validate key source files exist
required_files = [
    PROJECT_ROOT / "src" / "rag" / "knowledge_base.py",
    PROJECT_ROOT / "src" / "rag" / "vector_store.py",
    PROJECT_ROOT / "config" / "settings.py",
]
for f in required_files:
    status = "✅" if f.exists() else "❌ MISSING"
    print(f"  {status}  {f.relative_to(PROJECT_ROOT)}")

📁 Project root  : C:\santosh\MediGemma
📁 Working dir   : c:\santosh\MediGemma\notebooks
🐍 Python version: 3.14.2
  ✅  src\rag\knowledge_base.py
  ✅  src\rag\vector_store.py
  ✅  config\settings.py


## 🔍 Cell 3 — Library Version Check

In [4]:
import importlib

packages = {
    "chromadb": "chromadb",
    "langchain": "langchain",
    "langchain_community": "langchain_community",
    "langchain_huggingface": "langchain_huggingface",
    "sentence_transformers": "sentence_transformers",
    "torch": "torch",
}

print("📦 Installed package versions:")
print("-" * 40)
all_ok = True
for display_name, import_name in packages.items():
    try:
        mod = importlib.import_module(import_name)
        version = getattr(mod, "__version__", "unknown")
        print(f"  ✅ {display_name:<30} {version}")
    except ImportError:
        print(f"  ❌ {display_name:<30} NOT INSTALLED")
        all_ok = False

print("-" * 40)
print("✅ All packages ready" if all_ok else "❌ Some packages missing — re-run Cell 1")

📦 Installed package versions:
----------------------------------------
  ✅ chromadb                       1.5.9
  ✅ langchain                      1.3.1
  ✅ langchain_community            0.4.2


c:\santosh\MediGemma\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  ✅ langchain_huggingface          unknown
  ✅ sentence_transformers          5.5.1
  ✅ torch                          2.12.0+cpu
----------------------------------------
✅ All packages ready


## 📚 Cell 5 — Ingest Documents into Knowledge Base

In [ ]:
from rag.knowledge_base import MedicalKnowledgeBase
from pathlib import Path

KB_ROOT = Path(PROJECT_ROOT) / "data" / "knowledge_base"

print("🏥 Starting MediGemma Production Vector Ingestion Engine...")
    
if not os.path.exists(KB_ROOT):
    print(f"❌ Error: Data directory target '{KB_ROOT}' does not exist on disk.")
    
else:
    # 1. Gather all medical files from local disk (TXT, MD, and now PDF!)
    target_files = []

    for extension in ["*.txt", "*.md", "*.pdf"]:
        target_files.extend(
            Path(KB_ROOT).rglob(extension)
        )

    target_files = [str(f) for f in target_files]

    if  target_files:
        print(f"🔍 Discovered {len(target_files)} resource file(s) scheduled for ingestion.")
        # Extract content and map advanced metadata structures using MedicalKnowledgeBase
        kb = MedicalKnowledgeBase(output_dir=str(KB_ROOT))
        kb.load_documents(target_files)

        # Verify markdown metadata extraction
        print(f"\nLoaded docs: {len(kb.documents)}")

        sample = kb.documents[0]

        print("\nMetadata:")
        print(sample["metadata"])
        

    else:  
        print(f"⚠️ No source documents found inside '{KB_ROOT}'. Drop your medical PDFs/TXTs here first!")



📂 Found 9 documents to ingest:
   - nhs_cpr_guide.txt
   - who_first_aid_basic.txt
   - mayo_common_medications.txt
   - medlineplus_antibiotics.txt
   - cdc_diabetes_symptoms.txt
   - nhs_chest_pain_symptoms.txt
   - who_hypertension_symptoms.txt
   - cdc_sepsis_triage.txt
   - who_emergency_triage.txt
✅ Loaded: nhs_cpr_guide
   Source    : NHS
   Domain    : cardiovascular
   Trust     : 8
   Priority  : 10
   Risk      : emergency
✅ Loaded: who_first_aid_basic
   Source    : WHO
   Domain    : general_medicine
   Trust     : 10
   Priority  : 10
   Risk      : emergency
✅ Loaded: mayo_common_medications
   Source    : Mayo Clinic
   Domain    : pharmacology
   Trust     : 8
   Priority  : 8
   Risk      : low
✅ Loaded: medlineplus_antibiotics
   Source    : MedlinePlus
   Domain    : infectious_disease
   Trust     : 8
   Priority  : 9
   Risk      : low
✅ Loaded: cdc_diabetes_symptoms
   Source    : CDC
   Domain    : diabetes
   Trust     : 10
   Priority  : 8
   Risk      : low
✅

## ✂️ Cell 6 — Generate Chunks

In [ ]:
# Generate chunks from loaded documents
dataclass_chunks = kb.generate_chunks()

if not dataclass_chunks:
    print("❌ Data pipeline aborting: No chunks generated from source material.")
    
print(f"\nGenerated {len(dataclass_chunks)} chunks")

print("\nExample chunk:\n")

print(dataclass_chunks[0].text)

print("\nMetadata:\n")

print(dataclass_chunks[0].metadata)

✅ Generated 12 chunks total

📝 Sample chunk preview (first 3):
------------------------------------------------------------
[Chunk 1]
  ID       : 3154fe2c-3eef-412d-84bb-ad694ea2005e_0
  Domain   : cardiovascular
  Priority : 10
  Trust    : 8
  Preview  : National Health Service — CPR Quick Reference Guide

CARDIOPULMONARY RESUSCITATION (CPR) — EMERGENCY PROCEDURE

Before s...

[Chunk 2]
  ID       : 3154fe2c-3eef-412d-84bb-ad694ea2005e_1
  Domain   : cardiovascular
  Priority : 10
  Trust    : 8
  Preview  : AED: Use as soon as available — follow voice prompts exactly.

Source: NHS/Resuscitation Council UK, 2023...

[Chunk 3]
  ID       : 766bcdc0-ce31-410a-9cc4-6486093e8fcd_0
  Domain   : general_medicine
  Priority : 10
  Trust    : 10
  Preview  : World Health Organization — Basic First Aid Protocols

BURNS
1. Cool the burn with cool (not upper respiratory tract inf...


🏥 MEDICAL RAG PIPELINE STATISTICS
Total Chunks: 12
Unique Domains: {'cardiovascular': 4, 'general_medicine': 4

## 🔑 Cell 8 — Initialise Vector Store & Embed Chunks

In [ ]:
from rag.vector_store import MedicalVectorStore, transform_kb_chunks_to_docs
from config.settings import EMBEDDING_MODEL

# Initialise the vector store (downloads BGE model on first run ~100MB)
print("⏳ Initialising MedicalVectorStore...")
print("   (Embedding model downloads on first run — may take 1–2 mins)")
print()

vs = MedicalVectorStore()

print("\n📊 Initial collection stats:")
stats = vs.get_collection_stats()
for k, v in stats.items():
    print(f"   {k:<25}: {v}")

c:\santosh\MediGemma\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-06-01 21:18:27,122 - INFO - 🔧 Loading embeddings: BAAI/bge-large-en-v1.5


⏳ Initialising MedicalVectorStore...
   (Embedding model downloads on first run — may take 1–2 mins)

🔍 Initializing embedding model: BAAI/bge-large-en-v1.5
🔧 Using device: auto


2026-06-01 21:19:05,705 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-06-01 21:19:05,721 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/modules.json "HTTP/1.1 200 OK"
2026-06-01 21:19:06,081 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-06-01 21:19:06,099 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-06-01 21:19:06,122 - INFO - Loading SentenceTransformer model from BAAI/bge-large-en-v1.5.
2026-06-01 21:19:06,394 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 3


📊 Initial collection stats:
   name                     : medical_kb
   count                    : 0
   metadata                 : {'description': 'Medical KB Collection: medical_kb'}
   embedding_model          : BAAI/bge-large-en-v1.5


## ⬆️ Cell 9 — Add Chunks to Vector Store

In [ ]:
from dataclasses import asdict

# 4. Map MedicalChunk dataclass array instantly to ChromaDB specifications
print("🔄 Standardizing dataclass fields to vector schema layout...")
formatted_docs = transform_kb_chunks_to_docs(dataclass_chunks)

print(f"📋 Prepared {len(formatted_docs)} documents for embedding")
print(f"   Sample doc keys: {list(formatted_docs[0].keys())}")
print()

print(f"📡 Writing vectors to persistent disk storage folder...")
added_count = vs.add_documents(formatted_docs)
print(
f"Collection size: "
f"{vs.collection.count()}"
)

print("\n" + "="*60)
print("🏆 SYSTEM INGESTION PERFORMANCE METRICS")
print("="*60)
print(f"Successfully Persisted: {added_count} distinct vector nodes")
print(f"Target Save Location  : {vs.persist_dir}")
print("="*60)

# Run structural distribution stats directly from your knowledge base engine
kb.print_statistics()


2026-06-01 21:38:30,645 - INFO - ⬆️ Adding 12 documents to vector store...


📋 Prepared 12 documents for embedding
   Sample doc keys: ['id', 'content', 'metadata']



2026-06-01 21:38:43,601 - INFO - ✅ Added 12 valid documents total



📊 Updated collection stats:
   name                     : medical_kb
   count                    : 12
   metadata                 : {'description': 'Medical KB Collection: medical_kb'}
   embedding_model          : BAAI/bge-large-en-v1.5


## 🔍 Cell 10 — RAG Pipeline Validation: Similarity Search

In [ ]:
# Test queries covering each knowledge category
TEST_QUERIES = [

    # Symptoms
    ("high fever and chills", None),
    ("severe headache", None),
    ("persistent cough", None),
    ("abdominal pain after eating", None),

    # Respiratory
    ("asthma attack treatment", None),
    ("pneumonia symptoms", None),

    # Cardiovascular
    ("hypertension treatment", None),
    ("heart attack emergency", None),

    # First Aid
    ("first aid burn treatment steps", None),
    ("nosebleed first aid", None),

    # Pediatric
    ("child fever dehydration", None),

    # Infectious Disease
    ("malaria symptoms", None),
    ("tuberculosis treatment", None),
]

print("=" * 65)
print("🔍 SIMILARITY SEARCH VALIDATION")
print("=" * 65)

all_passed = True
for query, where_filter in TEST_QUERIES:
    filter_label = f" [filter: {where_filter}]" if where_filter else ""
    print(f"\n📝 Query: '{query}'{filter_label}")
    print("-" * 55)
    
    results = vs.similarity_search(query, k=3, where=where_filter)
    
    if not results:
        print("  ❌ No results returned")
        all_passed = False
        continue
    
    for i, doc in enumerate(results):
        print(f"  [{i+1}] Source : {doc.metadata.get('source', 'unknown')}")
        print(f"       Domain : {doc.metadata.get('domain', 'unknown')}")
        print(f"       Trust  : {doc.metadata.get('trust_level', '?')}")
        print(f"       Title  : {doc.metadata.get('doc_title')}")
        print(f"       Text   : {doc.page_content[:100].strip()}...")
        print()

print("=" * 65)
print("✅ Similarity search validation PASSED" if all_passed else "⚠️  Some queries returned no results")

🔍 SIMILARITY SEARCH VALIDATION

📝 Query: 'hypertension blood pressure treatment'
-------------------------------------------------------
  [1] Source : WHO
       Domain : cardiovascular
       Trust  : 10
       Text   : World Health Organization — Hypertension Fact Sheet

Hypertension (hypertension) is a major risk fac...

  [2] Source : WHO
       Domain : general_medicine
       Trust  : 10
       Text   : BLEEDING
1. Apply direct pressure with a clean cloth
2. Elevate the injured limb above heart level i...

  [3] Source : Mayo Clinic
       Domain : pharmacology
       Trust  : 8
       Text   : Mayo Clinic — Common Medications Reference Guide

PARACETAMOL (ACETAMINOPHEN)
- Uses: Pain relief, f...


📝 Query: 'chest pain heart attack emergency symptoms'
-------------------------------------------------------
  [1] Source : NHS
       Domain : cardiovascular
       Trust  : 8
       Text   : National Health Service — Chest Pain Assessment Guide

Chest pain requires immediate evalua

## 🔀 Cell 11 — RAG Pipeline Validation: Hybrid Search (MMR)

In [ ]:
print("=" * 65)
print("🔀 HYBRID SEARCH (MMR) VALIDATION")
print("=" * 65)


hybrid_results = vs.hybrid_search(
    query,
    k=4,
    lambda_mult=0.7   # 0.7 = favour relevance; 0.3 = favour diversity
)

if hybrid_results:
    print(f"✅ Returned {len(hybrid_results)} diverse results (MMR)")
    print()
    for i, doc in enumerate(hybrid_results):
        print(f"  [{i+1}] Source : {doc.metadata.get('source', 'unknown')}")
        print(f"       Domain : {doc.metadata.get('domain', 'unknown')}")
        print(f"       Text   : {doc.page_content[:110].strip()}...")
        print()
    
    # Diversity check: confirm sources are not all identical
    sources = [doc.metadata.get('source') for doc in hybrid_results]
    unique_sources = len(set(sources))
    print(f"📊 Source diversity: {unique_sources}/{len(hybrid_results)} unique sources")
    diversity_ok = unique_sources > 1
    print("✅ Good diversity" if diversity_ok else "⚠️ Low diversity — consider lowering lambda_mult")
else:
    print("❌ Hybrid search returned no results")

🔀 HYBRID SEARCH (MMR) VALIDATION

📝 Query: 'emergency symptoms requiring immediate medical attention'
-------------------------------------------------------
✅ Returned 4 diverse results (MMR)

  [1] Source : CDC
       Domain : infectious_disease
       Text   : CDC — Sepsis Early Recognition & Triage Protocol

Sepsis is a life-threatening emergency caused by the body's...

  [2] Source : NHS
       Domain : cardiovascular
       Text   : National Health Service — Chest Pain Assessment Guide

Chest pain requires immediate evaluation. Do not ignore...

  [3] Source : WHO
       Domain : general_medicine
       Text   : BLEEDING
1. Apply direct pressure with a clean cloth
2. Elevate the injured limb above heart level if possible...

  [4] Source : CDC
       Domain : diabetes
       Text   : Centers for Disease Control — Diabetes Symptoms Guide

Type 2 Diabetes Mellitus: Symptoms & Early Detection

E...

📊 Source diversity: 3/4 unique sources
✅ Good diversity


## 🤖 Cell 12 — RAG Pipeline Validation: LangChain Retriever

In [ ]:
print("=" * 65)
print("🤖 LANGCHAIN RETRIEVER VALIDATION")
print("=" * 65)

# Get retriever with MMR settings
retriever = vs.get_retriever(search_kwargs={"k": 4, "fetch_k": 12, "lambda_mult": 0.6})

retriever_query = "steps to treat severe bleeding wound"
print(f"\n📝 Retriever query: '{retriever_query}'")
print("-" * 55)

try:
    # LangChain retriever uses .invoke()
    retrieved_docs = retriever.invoke(retriever_query)
    
    print(f"✅ Retrieved {len(retrieved_docs)} documents via retriever")
    print()
    
    for i, doc in enumerate(retrieved_docs):
        print(f"  [{i+1}] Source : {doc.metadata.get('source', 'unknown')}")
        print(f"       Text   : {doc.page_content[:120].strip()}...")
        print()
        
except Exception as e:
    print(f"❌ Retriever failed: {e}")

2026-06-01 21:52:32,174 - INFO - ✅ Retriever created with MMR search


🤖 LANGCHAIN RETRIEVER VALIDATION

📝 Retriever query: 'steps to treat severe bleeding wound'
-------------------------------------------------------
✅ Retrieved 4 documents via retriever

  [1] Source : WHO
       Text   : BLEEDING
1. Apply direct pressure with a clean cloth
2. Elevate the injured limb above heart level if possible
3. Mainta...

  [2] Source : WHO
       Text   : World Health Organization — Basic First Aid Protocols

BURNS
1. Cool the burn with cool (not upper respiratory tract inf...

  [3] Source : WHO
       Text   : World Health Organization — Emergency Triage Reference (START Protocol)

TRIAGE CATEGORIES

RED (Immediate) — Life-threa...

  [4] Source : MedlinePlus
       Text   : MedlinePlus — Antibiotic Use Guidelines

Antibiotics treat bacterial infections — they DO NOT work against viruses.

Whe...



## 🛡️ Cell 13 — Safety Layer: Disclaimer Prepend Test

Tests the bug fix: disclaimer should be **prepended** (before the text), not appended.

In [ ]:
print("=" * 65)
print("🛡️  SAFETY LAYER VALIDATION")
print("=" * 65)

# Simulate retrieved chunk from a high-risk document
test_cases = [
    {
        "text": "Crush injury with uncontrolled bleeding and absent radial pulse.",
        "metadata": {"risk_level": "emergency", "source": "who_triage"}
    },
    {
        "text": "Blood pressure reading of 185/120 mmHg detected.",
        "metadata": {"risk_level": "high", "source": "who_hypertension"}
    },
    {
        "text": "Ibuprofen can be taken with food for mild pain relief.",
        "metadata": {"risk_level": "low", "source": "mayo_medications"}
    },
]

# Bug fix: disclaimer should be PREPENDED, not appended
def apply_safety_layer_fixed(kb_instance, text: str, metadata: dict) -> str:
    """Fixed: disclaimer prepended so user sees warning before content."""
    disclaimer = kb_instance.get_runtime_disclaimer(metadata)
    return f"{disclaimer}{text}"  # ✅ Disclaimer FIRST

for i, case in enumerate(test_cases):
    result = apply_safety_layer_fixed(kb, case["text"], case["metadata"])
    risk = case['metadata']['risk_level']
    print(f"\n[Test {i+1}] Risk Level: {risk.upper()}")
    print("-" * 55)
    print(result)

# Verify disclaimer comes BEFORE content
result_check = apply_safety_layer_fixed(
    kb,
    "Some medical content here.",
    {"risk_level": "emergency"}
)
disclaimer_first = result_check.index("CALL") < result_check.index("Some")
print("\n" + "=" * 65)
print("✅ Disclaimer correctly appears BEFORE content" if disclaimer_first 
      else "❌ FAIL: Disclaimer appeared after content")

## 📊 Cell 15 — Full Pipeline End-to-End Test

In [ ]:
# 01_setup_knowledge_base.ipynb
import os
import sys
from glob import glob

KB_ROOT = Path(PROJECT_ROOT) / "data" / "knowledge_base"

# Import your architectural classes and the streamlined conversion layer
from rag.knowledge_base import MedicalKnowledgeBase
from rag.vector_store import MedicalVectorStore, transform_kb_chunks_to_docs

def run_multimodal_ingestion_engine(data_dir: str = str(KB_ROOT)):
    print("🏥 Starting MediGemma Production Vector Ingestion Engine...")
    
    if not os.path.exists(data_dir):
        print(f"❌ Error: Data directory target '{data_dir}' does not exist on disk.")
        return

    # 1. Gather all medical files from local disk (TXT, MD, and now PDF!)
    target_files = []

    for extension in ["*.txt", "*.md", "*.pdf"]:
        target_files.extend(
            Path(data_dir).rglob(extension)
        )

    target_files = [str(f) for f in target_files]
    
    if not target_files:
        print(f"⚠️ No source documents found inside '{data_dir}'. Drop your medical PDFs/TXTs here first!")
        return
        
    print(f"🔍 Discovered {len(target_files)} resource file(s) scheduled for ingestion.")

    # 2. Extract content and map advanced metadata structures using MedicalKnowledgeBase
    kb = MedicalKnowledgeBase(output_dir=data_dir)
    kb.load_documents(target_files)
    
    # Verify markdown metadata extraction
    print(f"\nLoaded docs: {len(kb.documents)}")

    sample = kb.documents[0]

    print("\nMetadata:")
    print(sample["metadata"])
    
    # 3. Use RecursiveCharacterTextSplitter rules to structure semantic chunks
    dataclass_chunks = kb.generate_chunks()
    print(
    f"\nGenerated {len(dataclass_chunks)} chunks")

    print("\nExample chunk:\n")

    print(dataclass_chunks[0].text)

    print("\nMetadata:\n")

    print(dataclass_chunks[0].metadata)
    if not dataclass_chunks:
        print("❌ Data pipeline aborting: No chunks generated from source material.")
        return
        
    # 4. Map MedicalChunk dataclass array instantly to ChromaDB specifications
    print("🔄 Standardizing dataclass fields to vector schema layout...")
    formatted_docs = transform_kb_chunks_to_docs(dataclass_chunks)

    # 5. Connect to local Chroma DB index and save to disk
    vs = MedicalVectorStore()
    
    # Optional check: If you want a completely fresh database write session, uncomment below:
    # vs.clear_all()
    
    print(f"📡 Writing vectors to persistent disk storage folder...")
    added_count = vs.add_documents(formatted_docs)
    print(
    f"Collection size: "
    f"{vs.collection.count()}"
)
    
    print("\n" + "="*60)
    print("🏆 SYSTEM INGESTION PERFORMANCE METRICS")
    print("="*60)
    print(f"Successfully Persisted: {added_count} distinct vector nodes")
    print(f"Target Save Location  : {vs.persist_dir}")
    print("="*60)
    
    # Run structural distribution stats directly from your knowledge base engine
    kb.print_statistics()

# Run the complete structural data ingestion task
run_multimodal_ingestion_engine()

print("\nTesting Retrieval")
print("-" * 50)

results = vs.hybrid_search(
    "I have fever and chills",
    k=3
)

for i, r in enumerate(results, 1):

    print(f"\nResult {i}")

    print(r.metadata)

    print(r.page_content[:300])

ModuleNotFoundError: No module named 'models'